# Paso 4 — Marcado y formatos de datos de Matriculator

Matriculator controlaría el acceso al parking de Cebanc. La cámara tomaría una foto, la IA leería la matrícula y el sistema consultaría los permisos. Los casos dudosos pasarían a la garita y se guardarían las entradas y salidas.

## a y b. Qué son y dónde se utilizarían

| Tipo | Qué es, para qué sirve y características | Uso en Matriculator |
|---|---|---|
| HTML | Lenguaje de marcado que estructura una página mediante etiquetas, como títulos, formularios y tablas. El navegador lo interpreta. CSS aporta el diseño y JavaScript la interacción. | Interfaz de la garita: muestra foto, matrícula, confianza, registros y botones de revisión. |
| XML | Lenguaje de marcado con etiquetas personalizadas. Representa datos jerárquicos y permite validar su estructura con esquemas. Es más extenso que JSON. | Intercambio de datos con sistemas externos (si se exige XML); no para comunicación principal. |
| JSON | Formato de texto para representar e intercambiar datos mediante objetos y listas. Admite cadenas, números, booleanos y valores nulos. Es ligero y fácil de leer desde distintos lenguajes. | Comunicación IA-servidor (imagen, matrícula, confianza, fecha) y respuestas del servidor a interfaz. |
| Markdown | Marcado sencillo para escribir títulos, listas, enlaces y bloques de código. Las tablas están disponibles en variantes como las usadas por GitHub y Jupyter. | README del repositorio y celdas explicativas de los notebooks. |
| CSV | Formato de texto con filas y columnas separadas por un delimitador, normalmente una coma. Suele incluir una cabecera; no representa bien estructuras anidadas y no guarda tipos de datos explícitos. | Exportar registros de acceso para analizarlos y guardar datos tabulares, como los de Google Trends. |

Usaríamos principalmente HTML para la interfaz, JSON para el intercambio de datos, CSV para exportaciones y Markdown para documentar. XML quedaría como opción si otro sistema lo pidiera. La foto se enviaría como fichero; en el JSON incluiríamos su referencia, no solo el nombre esperando que contenga la imagen.

## c.i. Librerías en Python

| Fichero | Librerías principales | Funciones o características |
|---|---|---|
| HTML | Beautiful Soup (`bs4`), `lxml` y `html.parser` | Beautiful Soup facilita buscar etiquetas. `lxml` permite parsear HTML/XML. `html.parser` está incluido en Python. `pandas.read_html()` sirve para extraer tablas. |
| CSV | `csv` y `pandas` | `csv.reader()` y `csv.DictReader()` leen filas. `pandas.read_csv()` crea un DataFrame. |
| JSON | `json` y `pandas` | `json.load()` lee un fichero; `json.loads()` lee una cadena. `pandas.read_json()` y `json_normalize()` ayudan con datos tabulares. |

En esta demo utilizamos Beautiful Soup, `csv` y `json`. Parsear significa leer la estructura del fichero para poder acceder a sus elementos desde el programa.

## c.ii. Librerías equivalentes en R

| Tipo de fichero | Librerías en R | Ejemplo de lectura |
|---|---|---|
| HTML | `rvest`, apoyada en `xml2` | `rvest::read_html("interfaz.html")` |
| CSV | `readr` o las funciones de R base | `readr::read_csv("accesos.csv")` o `read.csv("accesos.csv")` |
| JSON | `jsonlite` | `jsonlite::fromJSON("entrada.json")` |

Estas funciones de R son referencias; las siguientes celdas se ejecutan en Python.


## Preparación
Ejecuta las celdas en orden. Los ejemplos crean sus propios ficheros, así que no necesitas subir datos. Todos los datos son inventados.

In [6]:
# Instala Beautiful Soup solo si no está disponible.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("bs4") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "beautifulsoup4"])

from bs4 import BeautifulSoup
from pathlib import Path
import json
import csv

carpeta = Path("datos_demo_matriculator")
carpeta.mkdir(exist_ok=True)


## c.iii. Fichero HTML de la interfaz
Inventamos una pantalla de la garita con una lectura pendiente de revisión. Los botones son una representación: este HTML no controla ninguna barrera.

In [11]:
html = """<!DOCTYPE html>
<html lang="es">
<head><meta charset="UTF-8"><title>Matriculator - Garita</title></head>
<body>
  <h1>Acceso al parking de Cebanc</h1>
  <section id="lectura">
    <img src="coche_prueba.jpg" alt="Foto sintética del vehículo">
    <p>Matrícula: <span id="matricula">1234ABC</span></p>
    <p>Confianza: <span id="confianza">0.76</span></p>
    <p id="estado">Pendiente de revisión manual</p>
    <button id="autorizar" type="button">Autorizar acceso</button>
    <button id="denegar" type="button">Denegar acceso</button>
  </section>
  <table id="registros">
    <thead><tr><th>Matrícula</th><th>Entrada</th><th>Salida</th></tr></thead>
    <tbody><tr><td>5678DEF</td><td>2026-10-01 08:00</td><td>Pendiente</td></tr></tbody>
  </table>
</body>
</html>"""

ruta_html = carpeta / "interfaz.html"
ruta_html.write_text(html, encoding="utf-8")
print("Fichero creado:", ruta_html)


Fichero creado: datos_demo_matriculator/interfaz.html


In [8]:
with open(ruta_html, encoding="utf-8") as fichero:
    pagina = BeautifulSoup(fichero, "html.parser")

print("Título:", pagina.title.get_text())
print("Matrícula:", pagina.find(id="matricula").get_text())
print("Confianza:", float(pagina.find(id="confianza").get_text()))
print("Estado:", pagina.find(id="estado").get_text())

for boton in pagina.find_all("button"):
    print("Botón:", boton.get_text())

for fila in pagina.select("#registros tbody tr"):
    valores = [celda.get_text() for celda in fila.find_all("td")]
    print("Registro:", valores)


Título: Matriculator - Garita
Matrícula: 1234ABC
Confianza: 0.76
Estado: Pendiente de revisión manual
Botón: Autorizar acceso
Botón: Denegar acceso
Registro: ['5678DEF', '2026-10-01 08:00', 'Pendiente']


## c.iv. Fichero JSON de entrada sintética
Representa los datos que enviaría el servicio de IA tras leer una imagen de la cámara. La autorización se consultaría después en la base de datos; no la decide la confianza de la IA.

In [9]:
contenido_json = """{
  "id_evento": "PRUEBA-001",
  "camara": "entrada_parking",
  "fecha_hora": "2026-10-01T08:15:00+02:00",
  "imagen": "coche_prueba.jpg",
  "sintetico": true,
  "lectura": {
    "matricula": "1234ABC",
    "confianza": 0.76
  },
  "fecha_salida": null
}"""

ruta_json = carpeta / "entrada.json"
ruta_json.write_text(contenido_json, encoding="utf-8")

with open(ruta_json, encoding="utf-8") as fichero:
    entrada = json.load(fichero)

print("Evento:", entrada["id_evento"])
print("Imagen:", entrada["imagen"])
print("Matrícula:", entrada["lectura"]["matricula"])
print("Confianza:", entrada["lectura"]["confianza"])
print("Datos sintéticos:", entrada["sintetico"])
print("Fecha de salida:", entrada["fecha_salida"])

matricula = entrada["lectura"]["matricula"]
confianza = entrada["lectura"]["confianza"]
if matricula is None or confianza < 0.80:
    print("Resultado: enviar a revisión en la garita")
else:
    print("Resultado: consultar los permisos en la base de datos")


Evento: PRUEBA-001
Imagen: coche_prueba.jpg
Matrícula: 1234ABC
Confianza: 0.76
Datos sintéticos: True
Fecha de salida: None
Resultado: enviar a revisión en la garita


## Ejemplo sencillo de lectura CSV
Simulamos una exportación de accesos. Una salida vacía indica que todavía no se ha registrado la salida.

In [10]:
contenido_csv = """matricula,fecha_entrada,fecha_salida,resultado
5678DEF,2026-10-01T08:00:00+02:00,,Autorizado automáticamente
1234ABC,2026-10-01T08:16:00+02:00,,Autorizado manualmente
"""
ruta_csv = carpeta / "accesos.csv"
ruta_csv.write_text(contenido_csv, encoding="utf-8")

with open(ruta_csv, encoding="utf-8", newline="") as fichero:
    registros = list(csv.DictReader(fichero))

for registro in registros:
    print(registro["matricula"], "-", registro["resultado"])
print("Número de accesos:", len(registros))


5678DEF - Autorizado automáticamente
1234ABC - Autorizado manualmente
Número de accesos: 2


## Conclusión

HTML estructura la pantalla de la garita. JSON permite recibir los datos de una lectura de matrícula. CSV facilita analizar registros como una tabla. Markdown nos sirve para explicar el trabajo. XML sería una alternativa para integraciones concretas.

La demo muestra cómo leer ficheros; no implementa la cámara, el modelo de IA, la base de datos ni la barrera.

## Fuentes

Documentación consultada el 01/10/2026:
- [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Python: csv](https://docs.python.org/3/library/csv.html)
- [Python: json](https://docs.python.org/3/library/json.html)
- [pandas: lectura y escritura](https://pandas.pydata.org/docs/user_guide/io.html)
- [rvest: read_html](https://rvest.tidyverse.org/reference/read_html.html)
- [readr: read_delim y read_csv](https://readr.tidyverse.org/reference/read_delim.html)
- [jsonlite](https://cran.r-project.org/package=jsonlite)
